# Dimensionality Reduction

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_low_rank_matrix
from sklearn.decomposition import PCA as PCA2

**Dimensionality reduction** is a technique of transforming data from a high-dimensional space to a low-dimenstional space. The goal is to reduce the dimensionality while preserving as much of the information of the original data as possible. This helps by compressing the data, making it easier, faster and cheaper to process, store and work with. In a Machine Learning setting, it can also be used to find the most relevant features of the data, allowing for faster model training while preserving most of the information in the data.

## Principal Component Analysis

<img src="https://upload.wikimedia.org/wikipedia/commons/1/15/GaussianScatterPCA.png" width="400" height="auto"/>

**Principal Component Analysis** is a dimensionality reduction technique which identifies the *principal components*. We can think of these as lines along which the data has the most variance. PCA requires standardized data. If the features have wildly different scales, the features with larger scale will dominate the calculations, in particular the computation of the Covariance matrix, with covariance depending on the magnitude of the variables.

It's important to point out that PCA is not a feature selection technique. The principal components are not a subset of the original features, rather they're combinations of the original features that preserve much of the original information.

The general process is as follows:

1. Compute the Covariance matrix.

$$
C = \frac{1}{n-1} \sum_{i}^n (X_i - \bar{X})(X_i - \bar{X})^T
$$

2. Find the Eigenvalues and Eigenvectors of the Covariance matrix

Eigenvectors represent the directions in which the data varies the most. Eigenvalues represent the magnitudes of the variance in each direction.

3. Order the Eigenvalues from largest to smallest and arrange the corresponding Eigenvectors in the same order.
4. Select top $k$ principal components ($k$ largest Eigenvalues and their Eigenvectors)
5. Project the data ono the new subspace

This means compute $AX$ where $A$ is the matrix of the top $k$ Eigenvectors and $X$ is our data. The resulting matrix is the data with reduced dimensionality.

In [14]:
class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.eigenvals = np.array([])
        self.eigenvecs = np.array([])

    def fit(self, X):
        # center data
        X = self._center(X)
        # compute covariance matrix
        C = np.cov(X.T, ddof=1)
        # calculate eigenvalues and eigenvectors
        eigenvals, eigenvecs = np.linalg.eigh(C)
        # sort by eigenvalues from largest to smallest
        idx = np.argsort(eigenvals)[::-1]
        eigenvals = eigenvals[idx]
        eigenvecs = eigenvecs[:, idx]
        # save variables
        self.eigenvals = eigenvals[:self.n_components]
        self.eigenvecs = eigenvecs[:, :self.n_components]
        self.explained_variance_ratio = self.eigenvals / eigenvals.sum()
        return self

    def transform(self, X):
        X = self._center(X)
        return X @ self.eigenvecs

    def _center(self, X):
        return (X - np.mean(X, axis=0))

In [15]:
X = make_low_rank_matrix(n_samples=1000, n_features=10, effective_rank=4, tail_strength=0.5, random_state=1)
pca = PCA(n_components=4).fit(X)
pca2 = PCA2(n_components=4, svd_solver='covariance_eigh').fit(X)

In [16]:
pca.eigenvals, pca2.explained_variance_

(array([0.00100099, 0.00091736, 0.000749  , 0.00055893]),
 array([0.00100099, 0.00091736, 0.000749  , 0.00055893]))

In [17]:
pca.explained_variance_ratio, pca2.explained_variance_ratio_

(array([0.21313346, 0.19532697, 0.15947859, 0.11900849]),
 array([0.21313346, 0.19532697, 0.15947859, 0.11900849]))

In [18]:
pca.transform(X)

array([[-5.17468366e-02, -1.76920237e-02, -1.31799228e-02,
        -2.71839420e-02],
       [-4.65865139e-02, -6.31547461e-02, -6.66164526e-03,
        -1.18270346e-02],
       [ 3.52592747e-02,  3.42921529e-02,  2.29635254e-02,
         1.10222074e-02],
       ...,
       [-3.88102646e-02,  3.38249256e-02,  2.41693114e-02,
        -1.34978385e-02],
       [ 5.14842209e-02, -5.02676175e-02,  1.75677081e-02,
        -9.34633408e-03],
       [-6.13380344e-02,  7.61560826e-05, -2.09615160e-02,
        -1.07377594e-02]], shape=(1000, 4))

In [19]:
pca2.transform(X)

array([[ 5.17468366e-02,  1.76920237e-02, -1.31799228e-02,
         2.71839420e-02],
       [ 4.65865139e-02,  6.31547461e-02, -6.66164526e-03,
         1.18270346e-02],
       [-3.52592747e-02, -3.42921529e-02,  2.29635254e-02,
        -1.10222074e-02],
       ...,
       [ 3.88102646e-02, -3.38249256e-02,  2.41693114e-02,
         1.34978385e-02],
       [-5.14842209e-02,  5.02676175e-02,  1.75677081e-02,
         9.34633408e-03],
       [ 6.13380344e-02, -7.61560826e-05, -2.09615160e-02,
         1.07377594e-02]], shape=(1000, 4))